# 01 — Data Exploration
Understand the distribution of placements, traits, augments, and items in the Challenger dataset.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
df = pd.read_csv('../data/processed/match_features.csv')
print(f'Dataset: {df.shape[0]:,} participant records from {df["match_id"].nunique():,} matches')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Placement distribution
df['placement'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Placement Distribution')
axes[0].set_xlabel('Placement')

# Level efficiency by placement
df.boxplot(column='level_efficiency', by='placement', ax=axes[1])
axes[1].set_title('Level Efficiency by Placement')
plt.sca(axes[1]); plt.xlabel('Placement')

# Gold left by placement
df.boxplot(column='gold_left', by='placement', ax=axes[2])
axes[2].set_title('Gold Left at Elimination by Placement')
plt.sca(axes[2]); plt.xlabel('Placement')

plt.tight_layout()
plt.show()

In [ ]:
# Top traits by frequency and top-4 rate
trait_stats = (
    df.groupby('top_trait')
    .agg(games=('placement', 'count'), top4_rate=('top4', 'mean'), avg_place=('placement', 'mean'))
    .query('games >= 20')
    .sort_values('top4_rate', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(trait_stats.index, trait_stats['top4_rate'], color='coral')
ax.axvline(0.5, color='gray', linestyle='--', label='50% baseline')
ax.set_title('Top-4 Rate by Dominant Trait (Challenger, min 20 games)')
ax.set_xlabel('Top-4 Rate')
ax.legend()
plt.tight_layout()
plt.show()
trait_stats

In [ ]:
# Augment tier distribution
print('Augment tier averages by placement:')
print(df.groupby('placement')['augment_tier_avg'].mean().round(2))

# Correlation heatmap of numeric features
numeric_cols = ['placement', 'level', 'gold_left', 'level_efficiency',
                'augment_tier_avg', 'unit_star_avg', 'item_concentration',
                'active_trait_count', 'damage_to_players']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()